# 01 · Spot corpus: load, validate, describe

**Purpose.** Load the team-written English spot profiles, check every field, show corpus statistics and a map, and save a clean version that all later notebooks use.

**Inputs**
- `data/spots/spots.csv`: one row per spot, written by our team from cited public sources (schema below)

**Outputs**
- `data/processed/spots_clean.json`: validated profiles with list fields parsed
- `results/corpus_validation.csv`: every problem found (errors and warnings)
- `results/figures/spot_map.html` (interactive) and `results/figures/corpus_stats.png`

**Why validate so strictly?** The condition-fit score uses orientation, swell window, tide and skill fields directly. A typo such as an orientation of `27` instead of `270` silently ranks a spot as "offshore" when the wind is actually onshore. That kind of error is invisible in the ranking but fatal in evaluation, so we stop early instead.

> The template file ships with **3 PLACEHOLDER spots** whose values are invented test values, not facts. Replace them with real, sourced profiles. **[TEAM]**

In [ ]:
# ---- Configuration ----
import os
from pathlib import Path


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / "notebooks").is_dir() and (folder / "requirements.txt").is_file():
            return folder
    raise FileNotFoundError("Project root not found. Open the project folder in VS Code (File > Open Folder).")


ROOT = find_project_root()
DATA_DIR = ROOT / "data"
SPOTS_FILE = DATA_DIR / "spots" / "spots.csv"
PROCESSED_DIR = DATA_DIR / "processed"
CORPUS_FILE = PROCESSED_DIR / "spots_clean.json"
RESULTS_DIR = ROOT / "results"
FIG_DIR = RESULTS_DIR / "figures"

LIBRARY_MODE = os.environ.get("SURF_IR_LIBRARY_MODE", "0") == "1"
SAMPLE_N = None          # e.g. 10 = only the first 10 spots
SEED = 42
STRICT = True            # True = stop on validation errors (warnings never stop the notebook)

# Controlled vocabularies. [TEAM] agree on these before writing profiles.
REGIONS = ["north", "northeast", "east", "south", "southwest", "west", "islands"]
BREAK_TYPES = ["beach", "reef", "point", "rivermouth"]
BOTTOM_TYPES = ["sand", "reef", "rock", "cobble", "mixed"]
TIDE_VALUES = ["low", "mid", "high", "any"]
SKILL_LEVELS = ["beginner", "intermediate", "advanced"]
CROWD_LEVELS = ["low", "medium", "high", "unknown"]
TAIWAN_BBOX = {"lat": (21.5, 26.5), "lon": (119.0, 122.5)}   # main island, Penghu, Green Island, Orchid Island
MAX_FORECAST_POINT_KM = 30    # warn if the offshore forecast point is further than this from the beach
MIN_DESCRIPTION_WORDS = 30
MIN_SOURCES = 2

## Profile schema

| Field | Type | Meaning |
|---|---|---|
| `spot_id` | text | Unique ID, letters/digits/underscore (e.g. `YL_01`) |
| `name`, `alt_names` | text | English name; alternative spellings separated by `;` (romanisation variants matter for search) |
| `region`, `nearest_town` | text | One of `REGIONS`; nearest town in English |
| `lat`, `lon` | float | Beach location (WGS84) |
| `forecast_lat`, `forecast_lon` | float, optional | Offshore point for the marine forecast (when the beach falls in a land grid cell) |
| `buoy_station_id`, `tide_station_id` | text, optional | CWA station used to check this spot |
| `break_type`, `bottom_type` | text | From `BREAK_TYPES`, `BOTTOM_TYPES` |
| `orientation_deg` | 0–359 | Direction the beach **faces** (90 = faces east). Offshore wind comes *from* orientation + 180° |
| `swell_dir_min_deg`, `swell_dir_max_deg` | 0–359 | Swell window, the directions swell must come *from*, clockwise from min to max (may wrap past 360) |
| `ideal_tide` | list | Subset of `low; mid; high` or `any` |
| `skill_levels` | list | Who the spot suits: subset of `beginner; intermediate; advanced` |
| `spot_max_wave_m` | float, optional | Size above which the spot stops working (closes out); caps the skill thresholds |
| `crowd_level` | text | `low`, `medium`, `high` or `unknown` |
| `hazards`, `facilities` | list | `;`-separated, e.g. `rocks; rips` / `board rental; showers` |
| `description` | text | Free-text profile in our own words (≥ 30 words) |
| `sources` | list | `;`-separated URLs or references (≥ 2 per real spot) |
| `is_placeholder` | bool | `TRUE` only for template rows |

## 1. Load the CSV
List fields are stored as `;`-separated text so the file stays easy to edit in Excel or Google Sheets. We parse them into Python lists here.

In [ ]:
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

LIST_FIELDS = ["alt_names", "ideal_tide", "skill_levels", "hazards", "facilities", "sources"]
NUMERIC_FIELDS = ["lat", "lon", "forecast_lat", "forecast_lon", "orientation_deg",
                  "swell_dir_min_deg", "swell_dir_max_deg", "spot_max_wave_m"]


def split_list(value):
    if pd.isna(value) or str(value).strip() == "":
        return []
    return [v.strip() for v in str(value).split(";") if v.strip()]


def load_spots_csv(path=SPOTS_FILE, sample_n=SAMPLE_N):
    df = pd.read_csv(path, dtype=str, keep_default_na=False, na_values=[""])
    df.columns = [c.strip() for c in df.columns]
    for col in NUMERIC_FIELDS:
        df[col] = pd.to_numeric(df[col], errors="coerce") if col in df else np.nan
    for col in LIST_FIELDS:
        df[col] = df[col].apply(split_list) if col in df else [[] for _ in range(len(df))]
    for col in ["region", "break_type", "bottom_type", "crowd_level", "buoy_station_id", "tide_station_id",
                "nearest_town", "description", "is_placeholder"]:
        if col not in df:
            df[col] = ""
    for col in ["region", "break_type", "bottom_type", "crowd_level"]:
        df[col] = df[col].fillna("").str.strip().str.lower()
    df["ideal_tide"] = df["ideal_tide"].apply(lambda xs: [x.lower() for x in xs])
    df["skill_levels"] = df["skill_levels"].apply(lambda xs: [x.lower() for x in xs])
    df["is_placeholder"] = df["is_placeholder"].fillna("").str.upper().isin(["TRUE", "1", "YES"])
    for col in ["buoy_station_id", "tide_station_id", "nearest_town", "description"]:
        df[col] = df[col].fillna("").str.strip()
    return df.head(sample_n) if sample_n else df


if not LIBRARY_MODE:
    spots_raw = load_spots_csv()
    print(f"Loaded {len(spots_raw)} spots from {SPOTS_FILE.relative_to(ROOT)}")
    display(spots_raw[["spot_id", "name", "region", "break_type", "skill_levels", "is_placeholder"]])

## 2. Validate every field
Each check produces a row in an issue table: *errors* break the system (missing coordinates, impossible angles, unknown categories), *warnings* weaken it (too few sources, very short descriptions). Keeping the issues as a table means the team can fix them in one pass.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371.0 * 2 * np.arcsin(np.sqrt(a))


def validate_spots(df):
    issues = []

    def add(spot, field, severity, msg):
        issues.append({"spot_id": spot, "field": field, "severity": severity, "message": msg})

    for dup in df.loc[df["spot_id"].duplicated(), "spot_id"]:
        add(dup, "spot_id", "error", "duplicate spot_id")
    for r in df.itertuples():
        sid = r.spot_id
        if not re.fullmatch(r"[A-Za-z0-9_]+", str(sid)):
            add(sid, "spot_id", "error", "use letters, digits and underscore only")
        for col in ["name", "region", "break_type", "bottom_type", "description"]:
            if not str(getattr(r, col)).strip():
                add(sid, col, "error", "required field is empty")
        for col in ["lat", "lon"]:
            v, (lo, hi) = getattr(r, col), TAIWAN_BBOX[col]
            if pd.isna(v) or not (lo <= v <= hi):
                add(sid, col, "error", f"missing or outside Taiwan ({lo} to {hi})")
        for col in ["orientation_deg", "swell_dir_min_deg", "swell_dir_max_deg"]:
            v = getattr(r, col)
            if pd.isna(v) or not (0 <= v < 360):
                add(sid, col, "error", "must be a compass angle in [0, 360)")
        for col, allowed in [("region", REGIONS), ("break_type", BREAK_TYPES), ("bottom_type", BOTTOM_TYPES)]:
            if getattr(r, col) and getattr(r, col) not in allowed:
                add(sid, col, "error", f"'{getattr(r, col)}' not in {allowed}")
        if r.crowd_level and r.crowd_level not in CROWD_LEVELS:
            add(sid, "crowd_level", "error", f"'{r.crowd_level}' not in {CROWD_LEVELS}")
        if not r.ideal_tide or any(t not in TIDE_VALUES for t in r.ideal_tide):
            add(sid, "ideal_tide", "error", f"use values from {TIDE_VALUES}")
        if not r.skill_levels or any(s not in SKILL_LEVELS for s in r.skill_levels):
            add(sid, "skill_levels", "error", f"use values from {SKILL_LEVELS}")
        if pd.notna(r.spot_max_wave_m) and r.spot_max_wave_m <= 0:
            add(sid, "spot_max_wave_m", "error", "must be positive")
        if pd.notna(r.forecast_lat) and pd.notna(r.lat):
            dist = haversine_km(r.lat, r.lon, r.forecast_lat, r.forecast_lon)
            if dist > MAX_FORECAST_POINT_KM:
                add(sid, "forecast_lat", "warning", f"forecast point is {dist:.0f} km from the beach")
        if len(str(r.description).split()) < MIN_DESCRIPTION_WORDS:
            add(sid, "description", "warning", f"shorter than {MIN_DESCRIPTION_WORDS} words")
        if r.is_placeholder:
            add(sid, "is_placeholder", "warning", "PLACEHOLDER row: replace with a real, sourced profile")
        elif len(r.sources) < MIN_SOURCES:
            add(sid, "sources", "warning", f"fewer than {MIN_SOURCES} sources")
        if not r.buoy_station_id:
            add(sid, "buoy_station_id", "warning", "no CWA buoy mapped: observations cannot verify this spot")
    return pd.DataFrame(issues, columns=["spot_id", "field", "severity", "message"])


if not LIBRARY_MODE:
    issues = validate_spots(spots_raw)
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    issues.to_csv(RESULTS_DIR / "corpus_validation.csv", index=False)
    print(issues["severity"].value_counts().to_dict() if len(issues) else "No issues found.")
    display(issues)

In [ ]:
if not LIBRARY_MODE:
    n_errors = int((issues["severity"] == "error").sum())
    if n_errors and STRICT:
        raise ValueError(f"{n_errors} validation errors: fix data/spots/spots.csv (see table above) and re-run.")

## 3. Corpus statistics
These numbers go into the *Domain & Data* section of the report: coverage per region and break type shows whether the corpus can answer the query types in our test collection (e.g. enough beginner spots in the south?).

In [ ]:
if not LIBRARY_MODE:
    spots = spots_raw.copy()
    spots["description_words"] = spots["description"].str.split().str.len()
    stats = {
        "spots": len(spots),
        "placeholders": int(spots["is_placeholder"].sum()),
        "with_buoy": int((spots["buoy_station_id"] != "").sum()),
        "mean_description_words": round(float(spots["description_words"].mean()), 1),
        "mean_sources": round(float(spots["sources"].str.len().mean()), 1),
    }
    print(stats)
    display(spots["region"].value_counts().rename("spots per region").to_frame())
    display(spots.explode("skill_levels")["skill_levels"].value_counts().rename("spots per skill level").to_frame())

In [ ]:
if not LIBRARY_MODE:
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
    spots["region"].value_counts().plot.bar(ax=axes[0], color="#4C78A8", title="Spots per region")
    spots["break_type"].value_counts().plot.bar(ax=axes[1], color="#F58518", title="Break type")
    spots.explode("skill_levels")["skill_levels"].value_counts().reindex(SKILL_LEVELS).plot.bar(
        ax=axes[2], color="#54A24B", title="Suitable skill level")
    for ax in axes:
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=30)
    fig.tight_layout()
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / "corpus_stats.png", dpi=200)
    plt.show()

## 4. Map of spots
The interactive map shows each spot with an arrow in the direction the beach faces. This is a quick visual check of `orientation_deg`: an east-coast beach should point east, out to sea. For the report, take a screenshot of the HTML map.

In [ ]:
import folium


def spot_map(df):
    m = folium.Map(location=[23.7, 121.0], zoom_start=7, tiles="OpenStreetMap")
    for r in df.itertuples():
        color = "gray" if r.is_placeholder else "blue"
        popup = f"<b>{r.name}</b><br>{r.region} · {r.break_type} · faces {r.orientation_deg:.0f}°<br>{', '.join(r.skill_levels)}"
        folium.CircleMarker([r.lat, r.lon], radius=6, color=color, fill=True, popup=popup, tooltip=r.name).add_to(m)
        end = [r.lat + 0.08 * np.cos(np.radians(r.orientation_deg)), r.lon + 0.08 * np.sin(np.radians(r.orientation_deg))]
        folium.PolyLine([[r.lat, r.lon], end], color=color, weight=3).add_to(m)
    return m


if not LIBRARY_MODE:
    m = spot_map(spots)
    m.save(str(FIG_DIR / "spot_map.html"))
    display(m)

## 5. Save the clean corpus
Later notebooks read only this JSON file, never the raw CSV, so they always see validated data with lists already parsed.

In [ ]:
def load_corpus(path=CORPUS_FILE):
    """Used by later notebooks: returns the validated corpus as a DataFrame."""
    if not path.exists():
        raise FileNotFoundError(f"{path} not found. Run 01_spot_corpus.ipynb first.")
    return pd.DataFrame(json.loads(path.read_text(encoding="utf-8")))


if not LIBRARY_MODE:
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    clean = spots_raw.replace({np.nan: None})
    CORPUS_FILE.write_text(json.dumps(clean.to_dict(orient="records"), indent=1, ensure_ascii=False), encoding="utf-8")
    print(f"Saved {len(clean)} spots to {CORPUS_FILE.relative_to(ROOT)}")